In [ ]:
import cv2
import numpy as np
import streamlit as st

# Function to process the images
def process_images(target, pattern):
    h, w = target.shape[:2]

    # === Perspective warp (adjust points accordingly)
    src_pts = np.float32([[280, 90], [290, 650], [980, 810], [1030, 250]])
    dst_pts = np.float32([[0, 0], [pattern.shape[1], 0],
                          [pattern.shape[1], pattern.shape[0]], [0, pattern.shape[0]]])
    matrix = cv2.getPerspectiveTransform(dst_pts, src_pts)
    warped = cv2.warpPerspective(pattern, matrix, (w, h))

    # === Create refined feathered alpha mask
    gray_warped = cv2.cvtColor(warped, cv2.COLOR_BGR2GRAY)
    _, mask = cv2.threshold(gray_warped, 1, 255, cv2.THRESH_BINARY)

    # Distance-based soft mask with better edge transition
    dist = cv2.distanceTransform(mask, cv2.DIST_L2, 5)
    alpha = cv2.normalize(dist, None, 0, 1.0, cv2.NORM_MINMAX)
    alpha = cv2.GaussianBlur(alpha, (31, 31), 0)   # Wider blur for softer edge
    alpha = np.power(alpha, 1.5)                   # Steepen falloff near edges

    # === Extract fold deformation from gradients
    gray = cv2.cvtColor(target, cv2.COLOR_BGR2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(gray)

    dx = cv2.Sobel(enhanced, cv2.CV_32F, 1, 0, ksize=5)
    dy = cv2.Sobel(enhanced, cv2.CV_32F, 0, 1, ksize=5)
    dx = cv2.GaussianBlur(dx, (9, 9), 0)
    dy = cv2.GaussianBlur(dy, (9, 9), 0)

    map_x, map_y = np.meshgrid(np.arange(w), np.arange(h))
    map_x = map_x.astype(np.float32) + 3 * dx / 255.0
    map_y = map_y.astype(np.float32) + 3 * dy / 255.0

    deformed = cv2.remap(warped, map_x, map_y, interpolation=cv2.INTER_LINEAR)

    # === Simulate soft shading
    luminance = cv2.GaussianBlur(enhanced, (15, 15), 0).astype(np.float32) / 255.0
    shading = np.clip(0.6 + luminance, 0.5, 1.5)
    shaded = np.clip(deformed.astype(np.float32) * shading[..., None], 0, 255)

    # === Blend pattern with flag using multiply and soft alpha
    flag_float = target.astype(np.float32)
    blended = (shaded / 255.0) * (flag_float / 255.0)
    blended = blended * 255

    # === Final alpha blend (with improved edge transition)
    opacity = 0.95
    final = alpha[..., None] * blended * opacity + (1 - alpha[..., None] * opacity) * flag_float
    final = np.clip(final, 0, 255).astype(np.uint8)

    return final

# Streamlit UI
st.title("Flag Pattern Warping and Blending")

# === File upload for images ===
flag_file = st.file_uploader("Upload Flag Image", type=["png", "jpg", "jpeg"])
pattern_file = st.file_uploader("Upload Pattern Image", type=["png", "jpg", "jpeg"])

if flag_file and pattern_file:
    # Read the uploaded images
    target = cv2.imdecode(np.frombuffer(flag_file.read(), np.uint8), cv2.IMREAD_COLOR)
    pattern = cv2.imdecode(np.frombuffer(pattern_file.read(), np.uint8), cv2.IMREAD_COLOR)

    # Process the images
    final_image = process_images(target, pattern)

    # Display the final image in Streamlit
    st.image(final_image, channels="BGR", use_column_width=True)
else:
    st.warning("Please upload both flag and pattern images.")
